# PrimeVul unseen-project Semgrep corroboration

This notebook runs the existing Semgrep corroboration protocol over the 38,362-record unseen-project Qwen output. It does not modify the raw PrimeVul JSONL or refit any model/policy.

Upload these files into the same Drive folder before running: `primevul_train.jsonl`, `qwen_unseen_project_results.jsonl`, `run_semgrep_corroboration.py`, and `semgrep_primevul_rules.yml`.

In [ ]:
!pip -q install semgrep==1.163.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path('/content/drive/MyDrive/primevul')
DATA = PROJECT_DIR / 'primevul_train.jsonl'
LLM = PROJECT_DIR / 'qwen_unseen_project_results.jsonl'
SCRIPT = PROJECT_DIR / 'run_semgrep_corroboration.py'
RULES = PROJECT_DIR / 'semgrep_primevul_rules.yml'
OUTPUT = PROJECT_DIR / 'semgrep_unseen_project_results.json'
for path in (DATA, LLM, SCRIPT, RULES):
    if not path.exists():
        raise FileNotFoundError(path)
print('Data:', DATA)
print('Qwen:', LLM)
print('Rules:', RULES)
print('Output:', OUTPUT)

In [ ]:
!python {SCRIPT} --data {DATA} --llm-results {LLM} --rules {RULES} --output {OUTPUT} --limit 0 --batch-size 512

In [ ]:
result = json.loads(OUTPUT.read_text(encoding='utf-8'))
print(json.dumps({
    'status': result.get('status'),
    'records_processed': result.get('records_processed'),
    'semgrep_version': result.get('semgrep_version'),
    'finding_records': result.get('summary', {}).get('finding_records'),
    'relations': result.get('summary', {}).get('relations'),
    'output': str(OUTPUT),
}, indent=2))